# ROI Extraction and Orthorectification

**Consolidates**: `chip_ortho.ipynb` + `point_roi_ortho.ipynb`  
**Source**: `grdl/example/ortho/`

## Learning Objectives

- Extract region-of-interest (ROI) chips centered on geographic points
- Two extraction patterns: **ground-extent** (fixed meters) vs **radius-based** (distance around point)
- Orthorectify to ENU meter grids or UTM projected grids
- Understand when to use each approach

## Theory: Extraction Pattern Decision Guide

| Use Case | Pattern | Grid Type | When to Use |
|----------|---------|-----------|-------------|
| **Fixed ground extent** | Ground-extent | ENUGrid (local meters) | Consistent output size; ML training datasets; multi-scale analysis |
| **Target-centered analysis** | Radius-based | UTMGrid (projected) | Detection cued by lat/lon; known target location; circular search area |

**Ground-extent example**: Extract 500m × 500m chips for a vehicle detection dataset (all chips same size).  
**Radius-based example**: Extract 250m radius around a suspected building for detailed inspection.

## Data Requirements

- SAR file (SICD or SIDD format)
- Target lat/lon coordinates (or use image center as fallback)

This notebook demonstrates both patterns on the same data.

## Imports & Environment Validation

In [ ]:
"""GRDL Notebook Preamble — Environment validation and autoreload."""
try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except (NameError, AttributeError):
    pass

import grdl
from packaging.version import Version

REQUIRED_VERSION = '0.6.1'
current_version = Version(grdl.__version__)
required_version = Version(REQUIRED_VERSION)

if current_version < required_version:
    raise RuntimeError(
        f"GRDL {REQUIRED_VERSION}+ required. Found {grdl.__version__}. "
        f"Update with: pip install --upgrade grdl"
    )

print(f"✓ GRDL {grdl.__version__} (>= {REQUIRED_VERSION})")

In [ ]:
from pathlib import Path
import gc
import numpy as np
import napari

from grdl.IO import get_reader
from grdl.geolocation import create_geolocation
from grdl.geolocation.chip import ChipGeolocation
from grdl.image_processing.ortho import orthorectify, ENUGrid, UTMGrid
from grdl.contrast import PercentileStretch

## Configuration

In [ ]:
# Input file
SAR_PATH = Path("/data/sar/from_duane/SICD/2024-01-13-15-37-10_UMBRA-05_SICD.nitf")#Path("/path/to/your/sicd_or_sidd_file.nitf")

# ROI center (or None to use image center)
CENTER_LAT = 34.05  # Example: Los Angeles area
CENTER_LON = -118.25

# Pattern 1: Ground-extent (fixed size in meters)
GROUND_EXTENT_M = 500.0  # Output will be exactly this size
GROUND_PIXEL_SIZE_M = 1.0  # ENU grid resolution

# Pattern 2: Radius-based (distance around point)
RADIUS_M = 250.0  # Circular region radius
UTM_PIXEL_SIZE_M = 0.5  # UTM grid resolution (finer detail)

# Validate file exists
if not SAR_PATH.exists():
    raise FileNotFoundError(f"File not found: {SAR_PATH}")

print(f"✓ Input file: {SAR_PATH.name}")
if CENTER_LAT is not None:
    print(f"  ROI center: ({CENTER_LAT:.4f}, {CENTER_LON:.4f})")
else:
    print(f"  ROI center: Image center (will be computed)")
print(f"\nPattern 1 (Ground-extent): {GROUND_EXTENT_M}m × {GROUND_EXTENT_M}m @ {GROUND_PIXEL_SIZE_M}m/px")
print(f"Pattern 2 (Radius-based): {RADIUS_M}m radius @ {UTM_PIXEL_SIZE_M}m/px")

## Common Setup: Determine ROI Center in Pixel Coordinates

Both patterns need the pixel coordinates of the ROI center. If `CENTER_LAT`/`CENTER_LON` are provided, we use `latlon_to_image()`. Otherwise, use the image center pixel.

In [ ]:
# CRITICAL: Use context manager to prevent file locks
with get_reader('sicd', SAR_PATH) as reader:
    meta = reader.metadata
    geo_full = create_geolocation(reader)
    
    rows, cols = reader.get_shape()[:2]
    print(f"Image size: {rows} × {cols} px")
    
    if CENTER_LAT is not None:
        # Convert geographic point to pixel coordinates
        result = geo_full.latlon_to_image(CENTER_LAT, CENTER_LON)
        center_row_candidate = float(result[0])
        center_col_candidate = float(result[1])
        
        # Check if coordinates are valid (within image and not NaN)
        if (np.isnan(center_row_candidate) or np.isnan(center_col_candidate) or
            center_row_candidate < 0 or center_row_candidate >= rows or
            center_col_candidate < 0 or center_col_candidate >= cols):
            print(f"⚠ WARNING: Requested coordinates ({CENTER_LAT:.4f}, {CENTER_LON:.4f}) are outside image bounds")
            print(f"  Falling back to image center")
            center_row = rows // 2
            center_col = cols // 2
            CENTER_LAT, CENTER_LON, _ = geo_full.image_to_latlon(center_row, center_col)
            print(f"  Image center: ({CENTER_LAT:.4f}, {CENTER_LON:.4f})")
        else:
            center_row = center_row_candidate
            center_col = center_col_candidate
            print(f"ROI center in pixels: ({center_row:.1f}, {center_col:.1f})")
    else:
        # Use image center
        center_row = rows // 2
        center_col = cols // 2
        CENTER_LAT, CENTER_LON, _ = geo_full.image_to_latlon(center_row, center_col)
        print(f"Using image center: ({CENTER_LAT:.4f}, {CENTER_LON:.4f})")
        print(f"  Pixel coordinates: ({center_row}, {center_col})")
    
    # Estimate pixel spacing (for chip size calculation)
    try:
        pixel_spacing_m = meta.grid.row.ss  # SICD row sample spacing
    except AttributeError:
        # Fallback: estimate from geolocation (sample 10px spacing)
        lat1, lon1, _ = geo_full.image_to_latlon(center_row, center_col)
        lat2, lon2, _ = geo_full.image_to_latlon(center_row + 10, center_col)
        from grdl.geolocation.utils import haversine_distance
        dist_10px = haversine_distance(lat1, lon1, lat2, lon2)
        pixel_spacing_m = dist_10px / 10.0
    
    print(f"Approximate pixel spacing: {pixel_spacing_m:.2f} m/px")

## Pattern 1: Ground-Extent Extraction (Fixed Size)

**Use case**: Consistent output dimensions for ML datasets, multi-scale analysis.  
**Grid type**: ENUGrid (East-North-Up local meters)  
**Key property**: Output is always exactly `GROUND_EXTENT_M × GROUND_EXTENT_M` meters.

### Step 1.1: Compute Chip Bounds

In [ ]:
# Estimate pixel extent from ground extent
half_extent_px = int((GROUND_EXTENT_M / 2) / pixel_spacing_m)

r_start_extent = max(0, int(center_row - half_extent_px))
c_start_extent = max(0, int(center_col - half_extent_px))
r_end_extent = min(rows, int(center_row + half_extent_px))
c_end_extent = min(cols, int(center_col + half_extent_px))

print(f"\nPattern 1 — Ground-Extent Chip Bounds:")
print(f"  Row range: [{r_start_extent}:{r_end_extent}] ({r_end_extent - r_start_extent} px)")
print(f"  Col range: [{c_start_extent}:{c_end_extent}] ({c_end_extent - c_start_extent} px)")

# CRITICAL: Use context manager
with get_reader('sicd', SAR_PATH) as reader:
    chip_extent = reader.read_chip(
        row_start=r_start_extent,
        row_end=r_end_extent,
        col_start=c_start_extent,
        col_end=c_end_extent,
    )

print(f"  ✓ Extracted chip: {chip_extent.shape}")

### Step 1.2: Define ENU Meter Grid

In [ ]:
# Create ChipGeolocation to preserve accuracy
with get_reader('sicd', SAR_PATH) as reader:
    geo_full = create_geolocation(reader)
    chip_shape = (r_end_extent - r_start_extent, c_end_extent - c_start_extent)
    geo_extent = ChipGeolocation(geo_full, row_offset=r_start_extent, col_offset=c_start_extent, shape=chip_shape)

# Define ENU grid from chip geolocation
enu_grid = ENUGrid.from_geolocation(
    geolocation=geo_extent,
    pixel_size_m=GROUND_PIXEL_SIZE_M,
    ref_lat=CENTER_LAT,
    ref_lon=CENTER_LON,
)

print(f"\nENU Grid:")
print(f"  Shape: {enu_grid.rows} × {enu_grid.cols} px")
print(f"  Resolution: {GROUND_PIXEL_SIZE_M} m/px")
print(f"  Center: ({CENTER_LAT:.4f}, {CENTER_LON:.4f})")

### Step 1.3: Orthorectify to ENU Grid

In [ ]:
# Convert complex to magnitude if needed
if np.iscomplexobj(chip_extent):
    magnitude_extent = np.abs(chip_extent)
else:
    magnitude_extent = chip_extent

# Orthorectify
ortho_extent_result = orthorectify(
    source_array=magnitude_extent,
    geolocation=geo_extent,
    output_grid=enu_grid,
    interpolation='bilinear',
)

ortho_extent = ortho_extent_result.data

print(f"\n✓ Pattern 1 orthorectified:")
print(f"  Output shape: {ortho_extent.shape}")
print(f"  Valid pixels: {np.sum(~np.isnan(ortho_extent))} / {ortho_extent.size}")

## Pattern 2: Radius-Based Extraction (Target-Centered)

**Use case**: Known target location, detection cueing, circular search area.  
**Grid type**: UTMGrid (projected coordinates)  
**Key property**: Extracts circular region around point (diameter = 2 × radius).

### Step 2.1: Compute Chip Bounds from Radius

In [ ]:
# Compute pixel extent from radius
radius_px = int(RADIUS_M / pixel_spacing_m)

r_start_radius = max(0, int(center_row - radius_px))
c_start_radius = max(0, int(center_col - radius_px))
r_end_radius = min(rows, int(center_row + radius_px))
c_end_radius = min(cols, int(center_col + radius_px))

print(f"\nPattern 2 — Radius-Based Chip Bounds:")
print(f"  Radius: {RADIUS_M}m (~{radius_px} px)")
print(f"  Row range: [{r_start_radius}:{r_end_radius}] ({r_end_radius - r_start_radius} px)")
print(f"  Col range: [{c_start_radius}:{c_end_radius}] ({c_end_radius - c_start_radius} px)")

# CRITICAL: Use context manager
with get_reader('sicd', SAR_PATH) as reader:
    chip_radius = reader.read_chip(
        row_start=r_start_radius,
        row_end=r_end_radius,
        col_start=c_start_radius,
        col_end=c_end_radius,
    )

print(f"  ✓ Extracted chip: {chip_radius.shape}")

### Step 2.2: Define UTM Grid

In [ ]:
# Create ChipGeolocation
with get_reader('sicd', SAR_PATH) as reader:
    geo_full = create_geolocation(reader)
    chip_shape = (r_end_radius - r_start_radius, c_end_radius - c_start_radius)
    geo_radius = ChipGeolocation(geo_full, row_offset=r_start_radius, col_offset=c_start_radius, shape=chip_shape)

# Define UTM grid from chip geolocation
utm_grid = UTMGrid.from_geolocation(
    geolocation=geo_radius,
    pixel_size_m=UTM_PIXEL_SIZE_M,
)

print(f"\nUTM Grid:")
print(f"  Shape: {utm_grid.rows} × {utm_grid.cols} px")
print(f"  Resolution: {UTM_PIXEL_SIZE_M} m/px")
print(f"  Zone: {utm_grid.zone}{'N' if utm_grid.north else 'S'}")
print(f"  Center: ({CENTER_LAT:.4f}, {CENTER_LON:.4f})")

### Step 2.3: Orthorectify to UTM Grid

In [ ]:
# Convert complex to magnitude if needed
if np.iscomplexobj(chip_radius):
    magnitude_radius = np.abs(chip_radius)
else:
    magnitude_radius = chip_radius

# Orthorectify
ortho_radius_result = orthorectify(
    source_array=magnitude_radius,
    geolocation=geo_radius,
    output_grid=utm_grid,
    interpolation='bilinear',
)

ortho_radius = ortho_radius_result.data

print(f"\n✓ Pattern 2 orthorectified:")
print(f"  Output shape: {ortho_radius.shape}")
print(f"  Valid pixels: {np.sum(~np.isnan(ortho_radius))} / {ortho_radius.size}")

## Visualization — Compare Both Patterns

In [ ]:
viewer = napari.Viewer(title="ROI Extraction Patterns Comparison")

# Convert to dB and apply percentile stretch for display
# Each image gets its own stretch to handle different dynamic ranges and NaN values

# Pattern 1: Ground-extent (slant)
extent_db = 20 * np.log10(magnitude_extent + 1e-12)
extent_stretch = PercentileStretch(plow=2.0, phigh=98.0)
extent_stretched = extent_stretch.apply(extent_db)

# Pattern 1: Ground-extent (ortho)
ortho_extent_db = 20 * np.log10(np.nan_to_num(ortho_extent, nan=1e-12) + 1e-12)
ortho_extent_stretch = PercentileStretch(plow=2.0, phigh=98.0)
ortho_extent_stretched = ortho_extent_stretch.apply(ortho_extent_db)

# Pattern 2: Radius-based (slant)
radius_db = 20 * np.log10(magnitude_radius + 1e-12)
radius_stretch = PercentileStretch(plow=2.0, phigh=98.0)
radius_stretched = radius_stretch.apply(radius_db)

# Pattern 2: Radius-based (ortho)
ortho_radius_db = 20 * np.log10(np.nan_to_num(ortho_radius, nan=1e-12) + 1e-12)
ortho_radius_stretch = PercentileStretch(plow=2.0, phigh=98.0)
ortho_radius_stretched = ortho_radius_stretch.apply(ortho_radius_db)

# Add layers to viewer
viewer.add_image(
    extent_stretched,
    name="Pattern 1: Slant (Ground-Extent)",
    colormap="gray",
    contrast_limits=[0, 1],
    visible=False,
)
viewer.add_image(
    ortho_extent_stretched,
    name="Pattern 1: ENU Ortho (Fixed Size)",
    colormap="gray",
    contrast_limits=[0, 1],
    visible=True,
)

viewer.add_image(
    radius_stretched,
    name="Pattern 2: Slant (Radius-Based)",
    colormap="gray",
    contrast_limits=[0, 1],
    visible=False,
)
viewer.add_image(
    ortho_radius_stretched,
    name="Pattern 2: UTM Ortho (Target-Centered)",
    colormap="gray",
    contrast_limits=[0, 1],
    visible=True,
)

# Simplify viewer UI
try:
    viewer.window._qt_viewer.dockLayerControls.setVisible(False)
    viewer.window._qt_viewer.dockConsole.setVisible(False)
    viewer.window._qt_viewer.activityDock.setVisible(False)
except AttributeError:
    pass

print("\n✓ Napari viewer opened")
print("\nViewer tips:")
print("  - Toggle layers to compare ground-extent vs radius-based patterns")
print("  - Pattern 1 (ENU): Fixed output size for ML datasets")
print("  - Pattern 2 (UTM): Target-centered for detection workflows")
print("\nExecution paused — close the napari window to continue and free memory...")

napari.run()

print("\n✓ Viewer closed — cleaning up memory...")
del chip_extent, chip_radius, magnitude_extent, magnitude_radius
del ortho_extent, ortho_radius
del extent_db, ortho_extent_db, radius_db, ortho_radius_db
del extent_stretched, ortho_extent_stretched, radius_stretched, ortho_radius_stretched
del extent_stretch, ortho_extent_stretch, radius_stretch, ortho_radius_stretch
del geo_extent, geo_radius, viewer
gc.collect()
print("✓ Memory cleanup complete")

## Summary

**GRDL patterns demonstrated**:
- ✅ **Pattern 1 (Ground-extent)**: Fixed ground size → ENUGrid → consistent output dimensions
- ✅ **Pattern 2 (Radius-based)**: Circular ROI around target → UTMGrid → target-centered analysis
- ✅ `latlon_to_image()` — convert geographic points to pixel coordinates
- ✅ `ChipGeolocation(geo, row_offset, col_offset)` — preserve geolocation accuracy for sub-images
- ✅ `ENUGrid.from_center()` — local East-North-Up meter grid (ENU)
- ✅ `UTMGrid.from_center()` — projected UTM grid with automatic zone detection
- ✅ **Context manager usage** (`with open_reader(path)`) — prevent OS-level file locks

**Pattern selection guide**:

| Use Case | Choose Pattern | Reason |
|----------|----------------|--------|
| ML training dataset | Ground-extent (ENU) | All chips same size |
| Detection around known target | Radius-based (UTM) | Circular search area |
| Multi-scale pyramid | Ground-extent (ENU) | Consistent extent across scales |
| Cued search (from GMTI, etc.) | Radius-based (UTM) | Geographic point input |

**Next steps**:
- Add DEM terrain correction: provide `dem_path` to geolocation constructor
- Batch extraction: wrap in loop over target list
- Target detection: see `image_processing/csi_detection_overlay.ipynb`
- ROI-cued detection: see `shapes/cued_detection.ipynb`